# Chapter 8 · Observational Study Design in Health: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, a population of 24,000 operations in 12 hospitals that adopted an infection-prevention bundle at different times, through the quality directorate's questions; only the sections with a calculation appear (8.3, 8.5 to 8.9, 8.11, and 8.12). The book draws the figures; the notebook prints the same numbers. **Part B** gives the written solution of every chapter exercise (1 to 12; statements are in the book) and, for the exercises whose solution needs a calculation (3, 4, 5, 7, and 10), its code and result. The code is Python 3 with pandas. Run the cells in order.

## Part A. The case

**Data.** `data/population.csv`: 24,000 rows, one per operation, with the hospital, the month, the patient's diabetes and smoking (true and reported), the surgery class, the bundle (true and as recorded in the registry), the infection within 30 days (true and as classified by two assessors), and whether follow-up was complete. `data/hospitals.csv`: the 12 hospitals, with the share of contaminated operations and the month of adoption of the bundle (empty when never adopted). `data/glucose_pairs.csv`: paired glucose readings from a meter and a laboratory. Each cell carries the number of the book section it belongs to.

### 8.3 The population we built

*How many operations, how many infections, and when did each hospital adopt the bundle?*

In [1]:
# 1. Read the population of 24,000 operations in 12 hospitals (blank registry fields are kept as empty text, not as missing values)
import pandas as pd

df = pd.read_csv("data/population.csv", keep_default_na=False)
print("Operations:", len(df), "; share followed by infection within 30 days:", round((df["ssi_30d"] == "yes").mean(), 4))

# 2. The 12 hospitals: share of contaminated operations and month of adoption of the bundle (empty when it was never adopted)
hosp = pd.read_csv("data/hospitals.csv", keep_default_na=False)
print(hosp.to_string(index=False))

Operations: 24000 ; share followed by infection within 30 days: 0.1011
hospital  contaminated_share bundle_adoption_month
     H01                0.25                      
     H02                0.30                      
     H03                0.35                      
     H04                0.40                    21
     H05                0.45                    19
     H06                0.50                    17
     H07                0.55                    15
     H08                0.60                    13
     H09                0.65                    13
     H10                0.70                    11
     H11                0.75                     9
     H12                0.80                     7


### 8.5 The ecological study

*Do the hospitals that adopted the bundle have more or fewer infections?*

In [2]:
# 1. One row per hospital: the share of contaminated operations, of patients with the bundle, and of infections (in %)
flags = pd.DataFrame({"hospital": df["hospital"],
                      "contaminated": df["surgery_class"] == "contaminated",
                      "bundle": df["bundle"] == "yes",
                      "infection": df["ssi_30d"] == "yes"})
by_hospital = flags.groupby("hospital").mean()
print((100 * by_hospital).round(1).head(3))

# 2. The correlation across hospitals between the bundle share and the infection share
print("Correlation between the bundle share and the infection share:", round(by_hospital["bundle"].corr(by_hospital["infection"]), 2))

          contaminated  bundle  infection
hospital                                 
H01               26.2     0.0        8.2
H02               31.0     0.0        9.8
H03               33.2     0.0        7.8
Correlation between the bundle share and the infection share: 0.72


### 8.6 The cohort study

*Among the patients followed from the adoption of the bundle, is it associated with fewer infections?*

In [3]:
# 1. The cohort: the operations on or after the month each hospital adopted the bundle (99 for the three that never did)
adopt = {}
for row in hosp.itertuples():
    if row.bundle_adoption_month:
        adopt[row.hospital] = int(row.bundle_adoption_month)
    else:
        adopt[row.hospital] = 99
cohort = df[df["month"] >= df["hospital"].map(adopt)]

# 2. The two-by-two table of the bundle and infection in the cohort, and the risk ratio of the bundle
t = pd.crosstab(cohort["bundle"], cohort["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
risk_ratio = (a / (a + b)) / (c / (c + d))
print("Operations in the cohort:", len(cohort), "; cells a, b, c, d:", int(a), int(b), int(c), int(d), "; risk ratio:", round(risk_ratio, 2))

# 3. The risk ratio of the bundle within each combination of surgery class and diabetes
for cls in ["clean", "contaminated"]:
    for dm in ["yes", "no"]:
        stratum = cohort[(cohort["surgery_class"] == cls) & (cohort["diabetes"] == dm)]
        t = pd.crosstab(stratum["bundle"], stratum["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
        a, b = t.loc["yes"]
        c, d = t.loc["no"]
        print("Class", cls, "; diabetes", dm, ": risk ratio", round((a / (a + b)) / (c / (c + d)), 2))

Operations in the cohort: 8240 ; cells a, b, c, d: 444 5463 232 2101 ; risk ratio: 0.76
Class clean ; diabetes yes : risk ratio 0.36
Class clean ; diabetes no : risk ratio 0.45
Class contaminated ; diabetes yes : risk ratio 0.49
Class contaminated ; diabetes no : risk ratio 0.44


In [4]:
# 4. Only the patients with complete follow-up: the cells and the risk ratio
complete = cohort[cohort["followup_complete"] == "yes"]
t = pd.crosstab(complete["bundle"], complete["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
print("Complete records:", len(complete), "; cells a, b, c, d:", int(a), int(b), int(c), int(d), "; risk ratio:", round((a / (a + b)) / (c / (c + d)), 2))

# 5. The share of patients with and without complete follow-up, by bundle and infection
lost = pd.crosstab([cohort["bundle"], cohort["ssi_30d"]], cohort["followup_complete"], normalize="index")
print(lost.round(3))

Complete records: 7736 ; cells a, b, c, d: 406 5185 152 1993 ; risk ratio: 1.02
followup_complete     no    yes
bundle ssi_30d                 
no     no          0.051  0.949
       yes         0.345  0.655
yes    no          0.051  0.949
       yes         0.086  0.914


### 8.7 The case-control study

*What does a case-control study of the same data give, and what if the controls come from the wrong source?*

In [5]:
# 1. All the cases of infection in the cohort, and two controls per case drawn from the cohort patients without infection (seed 8)
cases = cohort[cohort["ssi_30d"] == "yes"]
controls = cohort[cohort["ssi_30d"] == "no"].sample(n=2 * len(cases), random_state=8)
sample = pd.concat([cases, controls])
t = pd.crosstab(sample["bundle"], sample["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
print("Cases:", len(cases), "; cells a, b, c, d:", int(a), int(b), int(c), int(d), "; odds ratio:", round(a * d / (b * c), 2))

# 2. The invalid comparison: the same number of controls drawn from the whole population, hospitals that never adopted the bundle included
wrong = df[df["ssi_30d"] == "no"].sample(n=2 * len(cases), random_state=8)
sample = pd.concat([cases, wrong])
t = pd.crosstab(sample["bundle"], sample["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
print("Invalid controls - cells a, b, c, d:", int(a), int(b), int(c), int(d), "; odds ratio:", round(a * d / (b * c), 2))

Cases: 676 ; cells a, b, c, d: 444 969 232 383 ; odds ratio: 0.76
Invalid controls - cells a, b, c, d: 444 337 232 1015 ; odds ratio: 5.76


### 8.8 Real-world data and registries

*What happens when the blanks of a registry are read as no treatment?*

In [6]:
# 1. Recode every blank of the registry bundle field as "no", and count the blanks first (the silent decision made visible)
registry = cohort.copy()
registry["bundle_registry"] = registry["bundle_recorded"].replace("", "no")
print("Blank fields:", (cohort["bundle_recorded"] == "").sum(), "; share of blanks among the patients who did receive the bundle:", round((cohort.loc[cohort["bundle"] == "yes", "bundle_recorded"] == "").mean(), 3))

# 2. The risk ratio of the bundle with the registry field
t = pd.crosstab(registry["bundle_registry"], registry["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
print("Risk ratio with the registry field:", round((a / (a + b)) / (c / (c + d)), 2))

Blank fields: 745 ; share of blanks among the patients who did receive the bundle: 0.126
Risk ratio with the registry field: 0.81


### 8.9 The three families of systematic error

*How do selection, information, and confounding change the estimate?*

In [7]:
# 1. The risk ratio of smoking for infection in the whole population, with true smoking and with reported smoking
for column in ["smoker", "smoker_reported"]:
    t = pd.crosstab(df[column], df["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
    a, b = t.loc["yes"]
    c, d = t.loc["no"]
    print("Risk ratio of smoking, using", column + ":", round((a / (a + b)) / (c / (c + d)), 2))

# 2. True against reported smoking: the misclassification table that an investigator does not see
print(pd.crosstab(df["smoker"], df["smoker_reported"]))

# 3. The risk ratio of the bundle in the cohort with the infection classified by assessor A
assessed = cohort.assign(ssi_30d=cohort["ssi_assessor_a"])
t = pd.crosstab(assessed["bundle"], assessed["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
print("Risk ratio of the bundle with assessor A:", round((a / (a + b)) / (c / (c + d)), 2))

Risk ratio of smoking, using smoker: 2.06
Risk ratio of smoking, using smoker_reported: 1.81
smoker_reported     no   yes
smoker                      
no               16232   357
yes               1557  5854
Risk ratio of the bundle with assessor A: 0.8


### 8.11 Agreement between raters: percent agreement and kappa

*How well do two assessors agree, and how valid is each?*

In [8]:
# 1. The table of the two assessors, and the observed agreement
t = pd.crosstab(df["ssi_assessor_a"], df["ssi_assessor_b"]).loc[["yes", "no"], ["yes", "no"]]
print(t)
n = len(df)
p_o = (t.loc["yes", "yes"] + t.loc["no", "no"]) / n

# 2. The agreement expected by chance from the marginal proportions of each assessor, and kappa
row = t.sum(axis=1) / n
col = t.sum(axis=0) / n
p_e = row["yes"] * col["yes"] + row["no"] * col["no"]
print("Observed agreement:", round(p_o, 3), "; expected by chance:", round(p_e, 3), "; kappa:", round((p_o - p_e) / (1 - p_e), 2))

# 3. The validity of assessor A against the known infection status: sensitivity and specificity
v = pd.crosstab(df["ssi_assessor_a"], df["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
print("Sensitivity of assessor A:", round(v.loc["yes", "yes"] / v["yes"].sum(), 3), "; specificity:", round(v.loc["no", "no"] / v["no"].sum(), 3))

ssi_assessor_b   yes     no
ssi_assessor_a             
yes             1721   1009
no               703  20567
Observed agreement: 0.929 ; expected by chance: 0.808 ; kappa: 0.63
Sensitivity of assessor A: 0.863 ; specificity: 0.971


### 8.12 Agreement between methods: Bland-Altman analysis

*Do the glucose meter and the laboratory agree?*

In [9]:
# 1. Paired glucose readings: the meter and the laboratory
g = pd.read_csv("data/glucose_pairs.csv")
diff = g["glucose_meter"] - g["glucose_lab"]
print("Correlation between the laboratory and the meter:", round(g["glucose_lab"].corr(g["glucose_meter"]), 2))

# 2. Bland-Altman: the mean difference and the limits of agreement (mean plus or minus 1.96 standard deviations)
print("Mean difference:", round(diff.mean(), 1), "; standard deviation:", round(diff.std(), 1), "; limits:", round(diff.mean() - 1.96 * diff.std(), 1), "to", round(diff.mean() + 1.96 * diff.std(), 1))

Correlation between the laboratory and the meter: 0.96
Mean difference: 6.6 ; standard deviation: 8.7 ; limits: -10.5 to 23.6


## Part B. Exercise solutions

### 8.16 Exercise 1. Early mobilization and pneumonia

Population: adults undergoing hip surgery in the hospital. Exposure:
mobilization within 24 hours of surgery, compared with later mobilization.
Outcome: pneumonia by a stated definition. Time: thirty days from surgery.
The target experiment would enroll eligible patients, assign early or late
mobilization by lot, and follow all for thirty days. A study from the ward's
records departs from it in at least two ways: mobilization is decided by the
team, so the frailest patients are mobilized late (confounding by
indication); and the clock starts at surgery for everyone, but a patient who
develops pneumonia on day 1 may have early mobilization postponed, so classification by eventual mobilization time can depend on an earlier
outcome. The target strategies and the handling of the 24-hour window
must be specified before follow-up; merely relabeling patients afterward
does not restore the randomized comparison.

### 8.16 Exercise 2. What a photograph can measure

The proportion of inpatients with an infected wound: yes, it is a prevalence
among those present. The thirty-day risk: no, it needs follow-up of all
operations, and the photograph over-represents long stays. The proportion of
that month's operations with the bundle: only if the bundle is recorded for
everyone operated that month, which is a register question, not a survey
question; the inpatients present are not the month's operations. Whether the
bundle lowers the risk: no, for both reasons above and because the snapshot
cannot order exposure and outcome in time.

### 8.16 Exercise 3. Diabetes across hospitals and within patients

The hospital-level correlation is 0.35. Diabetes was generated with the
same 25% probability in every hospital, so its hospital proportions
vary only through the realized sample, while infection also varies with
case mix and other hospital characteristics; the individual-level risk
ratio, 2.52, compares diabetic with non-diabetic patients directly. One
answers “do hospitals with more diabetics have more infections” and the
other “do diabetic patients get infected more often.”

In [10]:
# 1. Across hospitals: the correlation between the diabetes share and the infection share
flags = pd.DataFrame({"hospital": df["hospital"], "diabetes": df["diabetes"] == "yes", "infection": df["ssi_30d"] == "yes"})
by_h = flags.groupby("hospital").mean()
print("Correlation across hospitals:", round(by_h["diabetes"].corr(by_h["infection"]), 2))

# 2. Within patients: the risk ratio of diabetes for infection
t = pd.crosstab(df["diabetes"], df["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
print("Risk ratio of diabetes:", round((t.loc["yes", "yes"] / t.loc["yes"].sum()) / (t.loc["no", "yes"] / t.loc["no"].sum()), 2))

Correlation across hospitals: 0.35
Risk ratio of diabetes: 2.52


### 8.16 Exercise 4. Stratifying by hospital

Nine hospitals adopted the bundle; their ratios run from 0.54 to 1.18 and
only one lies below 0.60. Stratifying holds hospital membership fixed;
it does not make recipients and non-recipients comparable on patient
risk or calendar time within each hospital. The ratios also vary by
chance. Within this eligible cohort, stratification by surgery class and diabetes
controls the confounding encoded in the generator. Its estimates still
vary by chance and need not equal the generating effect exactly.

In [11]:
# 1. The risk ratio of the bundle within each hospital of the cohort
for h in sorted(cohort["hospital"].unique()):
    sub = cohort[cohort["hospital"] == h]
    t = pd.crosstab(sub["bundle"], sub["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
    print("Hospital", h + ": risk ratio", round((t.loc["yes", "yes"] / t.loc["yes"].sum()) / (t.loc["no", "yes"] / t.loc["no"].sum()), 2))

Hospital H04: risk ratio 1.18
Hospital H05: risk ratio 0.99
Hospital H06: risk ratio 0.79
Hospital H07: risk ratio 0.83
Hospital H08: risk ratio 0.61
Hospital H09: risk ratio 0.85
Hospital H10: risk ratio 0.79
Hospital H11: risk ratio 0.54
Hospital H12: risk ratio 0.74


### 8.16 Exercise 5. Tracing half the losses

Tracing this random half moves the ratio from 1.02 to 0.89, toward the
complete-cohort association of 0.76. There is no rule that tracing half
removes half the bias: a risk ratio combines changing numerators and
denominators nonlinearly, and the traced sample varies by chance.
The untraced group may still be systematically different. Tracing only
patients who are easy to contact need not have the same result.

In [12]:
# 1. Trace half of the patients lost to follow-up (seed 8) and add them to the complete records
lost = cohort[cohort["followup_complete"] == "no"]
traced = lost.sample(frac=0.5, random_state=8)
known = pd.concat([cohort[cohort["followup_complete"] == "yes"], traced])

# 2. The risk ratio of the bundle among the known records
t = pd.crosstab(known["bundle"], known["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
print("Known records:", len(known), "; risk ratio:", round((t.loc["yes", "yes"] / t.loc["yes"].sum()) / (t.loc["no", "yes"] / t.loc["no"].sum()), 2))

Known records: 7988 ; risk ratio: 0.89


### 8.16 Exercise 6. Controls from one outpatient clinic

The source population of the cases is the patients operated in the twelve
hospitals during the study period, since any of them could have become a
case. Outpatients of one clinic are not that population: they were not
operated, and they come from one catchment. If the exposure were more common
in that catchment, the controls would be more often exposed than the source
population, the odds ratio would be pushed toward or below 1 for a harmful
exposure, and the error would be selection bias, not repaired merely by adjusting the measured covariates. Better control
sampling or additional information about the selection mechanism is needed.

### 8.16 Exercise 7. The blank field

Reading blanks as “no” gives 0.81 and as “yes” gives 0.76. The second
happens to be right here because every blank belongs to a true recipient,
but the analyst cannot know that from the data; the data dictionary must say
what a blank means (not given, or not recorded), and if it cannot, the
analysis reports both.

In [13]:
# 1. Recode the blank registry field as "no" and then as "yes", and compare the risk ratios of the bundle
registry = cohort.copy()
for value in ["no", "yes"]:
    registry["b"] = registry["bundle_recorded"].replace("", value)
    t = pd.crosstab(registry["b"], registry["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
    print("Blanks recoded as", value + ": risk ratio", round((t.loc["yes", "yes"] / t.loc["yes"].sum()) / (t.loc["no", "yes"] / t.loc["no"].sum()), 2))

Blanks recoded as no: risk ratio 0.81
Blanks recoded as yes: risk ratio 0.76


### 8.16 Exercise 8. Three families

(a) Selection bias: volunteers differ from non-volunteers in ways related to
the outcome. (b) Information bias, differential (recall bias): the
exposure is measured more completely in cases. (c) Confounding by
indication: severity is associated with the prescription and affects
mortality. (d) Selection bias: loss related to both exposure and outcome.
(e) Differential information bias: blood pressure is the outcome, and its
measurement error depends on obesity, the exposure.

### 8.16 Exercise 9. Two nurses and 300 ulcers

Percent agreement $p_o=(42+228)/300=0.900$. Nurse A calls 60/300 = 0.200
present and nurse B 54/300 = 0.180, so
$p_e=0.200\times0.180+0.800\times0.820=0.692$ and
$\kappa=(0.900-0.692)/(1-0.692)=0.68$. Changing ulcer prevalence can change both the observed agreement and
agreement expected from the margins, even if the nurses' sensitivity and
specificity remain constant. The table does not identify the true prevalence
or either nurse's error rates. It therefore cannot determine the size or
direction of a change in kappa in a second ward.

### 8.16 Exercise 10. Forty milligrams added to every meter reading

The correlation is unchanged at 0.96; the mean difference moves from 6.6 to
46.6 mg/dL and the limits of agreement from $-10.5$ to $23.6$ to $29.5$ to
$63.6$; the standard deviation of the differences is unchanged. The modified meter reads 46.6 mg/dL high on average, with substantial
patient-to-patient variation, yet retains the same high correlation: correlation measures whether the two rise together, not
whether they agree.

In [14]:
# 1. Forty milligrams added to every meter reading: the correlation, the mean difference, and the limits of agreement
meter = g["glucose_meter"] + 40
diff = meter - g["glucose_lab"]
print("Correlation:", round(g["glucose_lab"].corr(meter), 2), "; mean difference:", round(diff.mean(), 1), "; standard deviation:", round(diff.std(), 1), "; limits:", round(diff.mean() - 1.96 * diff.std(), 1), "to", round(diff.mean() + 1.96 * diff.std(), 1))

Correlation: 0.96 ; mean difference: 46.6 ; standard deviation: 8.7 ; limits: 29.5 to 63.6


### 8.16 Exercise 11. Sensitivity and specificity of self-report

From the table, 5,854 of the 7,411 true smokers report smoking
(sensitivity 0.79) and 16,232 of the 16,589 non-smokers report not
smoking (specificity 0.98). The reported-smoker group therefore contains a
few non-smokers and, more important, the reported-non-smoker group contains
1,557 true smokers with a smoker's risk. Each group is a mixture of the
two true groups, and a mixture of two risks lies between them, so the
observed ratio, 1.81, lies between the full-data association of 2.06 and 1 in this simulation. This is not a
universal direction-of-bias rule for all measurement-error settings.

### 8.16 Exercise 12. Choosing the design

(a) Cross-sectional survey with measurement of glucose: prevalence of
undiagnosed diabetes. (b) Case-control study, since the reaction is rare:
odds ratio for the variant. (c) Cohort of night-shift workers and
day-shift workers, ideally from occupational records with long follow-up:
rates and rate ratios for each cancer. (d) An experiment across the
departments, the cluster-randomized design of Chapter 10;
failing that, a controlled before-and-after study: difference in mean
waiting time. (e) An agreement study with paired measurements on the same
patients: mean difference and limits of agreement, not a correlation.

---
*Synthetic data (`generate_data.py`, seed 8). Teaching values only; nothing here is an estimate for a real hospital network.*